# Embeddings And Similarity

Text similarity is the comparison problem at the heart of systems like search, recommendation, and RAG. In this notebook you will build this comparison from scratch, making every step from raw text to similarity score visible and explainable, so you can understand the workflow that runs inside every production embedding system.


## Learning Objectives

By the end of this notebook, you should be able to:

- Build TF-IDF-style text vectors.
- Compare a query with documents using cosine similarity.
- Inspect why a result ranked first.
- Explain why similarity scores are relative, not universal truth.


## Why Similarity Needs Vectors

Text cannot be compared mathematically in its raw form. Embeddings solve this by **turning text into a numeric representation** you can do arithmetic on. That vector holds a position in a representation space: texts with similar meanings are mapped to nearby points, capturing relationships like `king - man + woman = queen`. Embeddings are central to semantic search, recommendations, clustering, duplicate detection, routing, and RAG.

<div align="center">
  <img src="assets/03-word-vector-space.png" alt="Semantic Vector Space Word Embeddings" width="400" style="max-width: 100%; height: auto;" />
  <p><em>Semantic vector space relationships demonstrating linguistic analogies. Source: <a href="https://upload.wikimedia.org/wikipedia/commons/3/3f/Word_vector_illustration.jpg" target="_blank" rel="noopener noreferrer">Wikimedia Commons</a></em></p>
</div>


**From text to vectors.**

In modern LLM systems, embeddings are dense vectors produced by neural models. This notebook uses a **TF-IDF**-style vectorizer instead. A neural model is more powerful and can match meaning even when the wording differs entirely, but it works as a black box. Building the mechanics first makes the black box easier to reason about. The retrieval workflow is still the same regardless of which vectorizer you use: document preparation, vector creation, query vectorization, similarity scoring, ranking, and result inspection.

For a deeper dive into computing and comparing high-dimensional semantic vectors using pre-trained sentence encoders, refer to the [OpenAI Embeddings Guide](https://platform.openai.com/docs/guides/embeddings) and the [Sentence Transformers (SBERT) Documentation](https://sbert.net/). If you are using standard Python machine learning packages, see scikit-learn's documentation on the [TF-IDF Vectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) and [pairwise cosine similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html).


## From Keyword Matching Toward Semantic Search

Exact keywords often fail because the same idea can be expressed in different words. A developer searching for "how to reduce API costs" and a document explaining "optimizing token usage for budget efficiency" share the same intent but no vocabulary in common. A pure keyword system might miss that relationship and score them as unrelated.

Closing that gap requires comparing meaning rather than exact word matches. Dense **semantic search** systems usually do this with model-generated embeddings. This notebook builds the foundation: you will construct vectors, compare them with cosine similarity, and inspect the retrieval workflow from the inside. Notebook 04 keeps the workflow local and transparent with sparse vectors, then names the gap between the exercise and production semantic search.


```mermaid
graph LR
    A[Text documents] --> B[Preprocess]
    B --> C[Vectorize]
    C --> D[Compare with query]
    D --> E[Rank by similarity]
    E --> F[Inspect top results]
```


## What Cosine Similarity Means

Cosine similarity compares **vector direction** rather than raw length. Two texts receive a high score when their weighted features point in a similar direction, meaning the angle $\theta$ between them is small. This geometric vector alignment (visualized in the quadrants below) is useful because a long document should not carry more weight simply because it has more words.  

The score is also relative to the corpus, preprocessing choices, and vectorizer. A score of 0.35 can be strong in one setup and weak in another, so always inspect examples rather than treating the number as universal truth.

<div align="center">
  <img src="assets/03-cosine-similarity.png" alt="Cosine Similarity & Vector Alignment" width="450" style="max-width: 100%; height: auto;" />
  <p><em>Geometric vector alignment and cosine similarity quadrants. Source: <a href="https://upload.wikimedia.org/wikipedia/commons/2/23/CosineSimilarity.png" target="_blank" rel="noopener noreferrer">Wikimedia Commons / Vector Similarity Model</a></em></p>
</div>


## Where Popular Libraries Fit

This notebook builds vectors from scratch so the mechanics are visible. In real LLM projects, the same steps are usually handled by libraries or services:

| Need | Common tool category | Example |
| --- | --- | --- |
| Create local text embeddings | Embedding model library | [`sentence_transformers`](https://www.sbert.net/docs/quickstart.html) |
| Use provider or model-specific embedding APIs through one interface | Application framework integration | LangChain embedding integrations |
| Store and search many dense vectors efficiently | Vector search library | FAISS |
| Store documents, metadata, embeddings and retrieval state together | Vector database | Chroma |

Note that these tools do not replace the concepts in this notebook. They automate them. You still need to understand tokenization, document preparation, vector creation, similarity scoring, metadata, and evaluation to debug the system.


## Implement The Vectorizer

The following functions implement each step a library would normally hide: tokenize raw text, weight terms, build normalized vectors, and compare their direction.


### Tokenization

The first step is to break raw text into discrete units a vectorizer can count and weight. The helpers below come from the previous notebook and they are included here so the full pipeline runs in one place.


In [ ]:
import csv
import math
import random
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

TOKEN_PATTERN = re.compile(
    r"""
    [A-Za-z]+(?:'[A-Za-z]+)?  # Words and contractions
    |\d+(?:\.\d+)?            # Integers and decimals
    |[^\w\s]                  # Punctuation and symbols
    """,
    re.VERBOSE,
)


@dataclass(frozen=True)
class TokenizationResult:
    text: str
    tokens: list[str]

    @property
    def count(self) -> int:
        return len(self.tokens)


def simple_tokenize(text: str) -> list[str]:
    """Split text into approximate tokens."""
    return TOKEN_PATTERN.findall(text)


def inspect_text(text: str) -> TokenizationResult:
    return TokenizationResult(text=text, tokens=simple_tokenize(text))


def count_tokens(text: str) -> int:
    return len(simple_tokenize(text))


def token_budget_remaining(
    context_window: int, prompt_tokens: int, reserved_output_tokens: int
) -> int:
    return context_window - prompt_tokens - reserved_output_tokens

### Text cleanup and feature selection

Before a vector can be built, the tokens need to be filtered down to their most informative terms, and each remaining term needs a weight reflecting how rare it is across the corpus. The next two blocks define the data structures and functions for this process. The following section will use these weights to build and compare the vectors.


In [ ]:
# Sparse vectors store only terms that appear, keeping the representation readable.
SparseVector = dict[str, float]

# Remove common glue words so topic words carry more of the score.
STOPWORDS = {
    "a",
    "an",
    "and",
    "are",
    "as",
    "be",
    "by",
    "can",
    "do",
    "does",
    "for",
    "from",
    "how",
    "i",
    "in",
    "into",
    "is",
    "it",
    "me",
    "my",
    "of",
    "or",
    "should",
    "the",
    "this",
    "to",
    "what",
    "when",
    "whether",
    "why",
    "with",
}

`content_tokens` applies the stopword filter and lowercases each token. `combine_text_fields` joins multiple record fields into one searchable string so retrieval can use both metadata and content. `fit_idf` measures how rare each token is across documents: common terms receive lower weights, rare terms receive higher ones.


In [ ]:
# Keep fitted vector state together so search queries use the same vocabulary and weights.
@dataclass(frozen=True)
class EmbeddedCorpus:
    records: list[dict[str, str]]
    text_field: str
    idf: dict[str, float]
    vectors: list[SparseVector]


def content_tokens(text: str) -> list[str]:
    """Return comparable lowercase tokens after removing low-signal words."""
    tokens = [token.lower() for token in simple_tokenize(text)]
    return [
        token
        for token in tokens
        if token not in STOPWORDS and any(char.isalnum() for char in token)
    ]


def combine_text_fields(record: dict[str, str], fields: list[str]) -> str:
    """Join selected metadata fields into the text that will be vectorized."""
    missing = [field for field in fields if field not in record]
    if missing:
        raise ValueError(f"Missing expected fields: {missing}")
    return " ".join(record.get(field, "") for field in fields)


def fit_idf(texts: list[str]) -> dict[str, float]:
    """Learn how informative each token is across the corpus."""
    document_count = len(texts)
    document_frequency: Counter[str] = Counter()
    for text in texts:
        document_frequency.update(set(content_tokens(text)))
    return {
        token: math.log((1 + document_count) / (1 + frequency)) + 1
        for token, frequency in document_frequency.items()
    }

### Vector construction and comparison

With the document weights available, the next block builds the normalized vectors and compares their direction using cosine similarity. Keeping this separate from text cleanup makes the retrieval pipeline easier to debug.


`vectorize` combines term frequency with inverse document frequency and normalizes the result. `embed_corpus` applies this to every document at once, fitting weights across the full corpus. `cosine_similarity` compares two vectors by their shared dimensions. `similarity_scores` closes the pipeline by scoring a query against every document vector.


In [ ]:
def vectorize(text: str, idf: dict[str, float]) -> SparseVector:
    """Convert text into a normalized TF-IDF vector for cosine comparison."""
    tokens = content_tokens(text)
    if not tokens:
        return {}
    counts = Counter(tokens)
    total = len(tokens)
    vector = {
        token: (count / total) * idf.get(token, 0.0)
        for token, count in counts.items()
        if token in idf
    }
    norm = math.sqrt(sum(value * value for value in vector.values()))
    if norm == 0:
        return vector
    return {token: value / norm for token, value in vector.items()}


def embed_corpus(records: list[dict[str, str]], text_field: str) -> EmbeddedCorpus:
    """Fit vector weights once, then vectorize every record with the same weights."""
    if not records:
        raise ValueError("Cannot embed an empty corpus.")
    texts = [record[text_field] for record in records]
    idf = fit_idf(texts)
    vectors = [vectorize(text, idf) for text in texts]
    return EmbeddedCorpus(
        records=records, text_field=text_field, idf=idf, vectors=vectors
    )


def cosine_similarity(left: SparseVector, right: SparseVector) -> float:
    """Return a directional similarity score for two sparse vectors."""
    if not left or not right:
        return 0.0
    shared = set(left).intersection(right)
    return sum(left[token] * right[token] for token in shared)


def similarity_scores(corpus: EmbeddedCorpus, query: str) -> list[float]:
    """Score one query against every document vector in the fitted corpus."""
    query_vector = vectorize(query, corpus.idf)
    return [cosine_similarity(query_vector, vector) for vector in corpus.vectors]

## Load The Data

This notebooks uses a [catalog of LLM course lessons](data/course_catalog.csv), each covering a topic from the course. Each row is a small document that includes lesson ID, title, module, and description.

The code below combines title, module and description into one `search_text` field so retrieval can use both metadata and explanatory content. Keeping also the original fields allows later cells to display readable titles and modules instead of anonymous vectors.


In [ ]:
# Load local CSV data.
catalog_path = Path("data") / "course_catalog.csv"
with catalog_path.open(newline="", encoding="utf-8") as file:
    records = list(csv.DictReader(file))
for record in records:
    record["search_text"] = combine_text_fields(
        record, ["title", "module", "description"]
    )
print(f"Loaded {len(records)} course lessons")
for record in records:
    print(f"- {record['lesson_id']}: {record['title']}")

Notice that metadata is part of the system, not decoration. Titles, modules, source URLs, timestamps, and document IDs can help users and developers interpret why a result appeared.


## Build Text Vectors

The next cell fits IDF weights across the full catalog and vectorizes every document. Fitting first means every document and every query use the same weights, so similarity scores are comparable across the corpus.


In [ ]:
# Fit vectors once over the catalog before comparing individual queries.
corpus = embed_corpus(records, text_field="search_text")
print(f"Vocabulary size: {len(corpus.idf)}")
print("Sample terms:", random.sample(list(corpus.idf), 20))

## Rank Lessons For A Query

The next block runs one query through the full pipeline: every lesson in the corpus is scored, the results are sorted by descending similarity, and the top three are printed with their titles and scores.


In [ ]:
# Rank every catalog row against one query, then inspect the strongest matches.
query = "How can I estimate the price of an LLM request?"
scores = similarity_scores(corpus, query)
ranked = sorted(range(len(scores)), key=lambda index: scores[index], reverse=True)
print(f"Query: {query}\n")
for rank, index in enumerate(ranked[:3], start=1):
    row = records[index]
    print(f"{rank}. {row['title']} | score={scores[index]:.3f}")
    print(f"   {row['description'][:150]}...")

## Inspect Matching Terms

Sparse vector systems like TF-IDF are easier to inspect because shared content tokens are visible. Neural embeddings are less transparent, but inspecting top results is still required.


In [ ]:
# Compare query terms with the top document to inspect why it ranked first.
top_index = ranked[0]
query_terms = set(content_tokens(query))
doc_terms = set(content_tokens(records[top_index]["search_text"]))
print("Query terms:", sorted(query_terms))
print("Shared terms with top result:", sorted(query_terms & doc_terms))

## Embedding System Anatomy

```mermaid
graph LR
    A["Raw documents"] --> B["Clean and<br/>combine fields"]
    B --> C["Chunk or keep<br/>short records"]
    C --> D["Create vectors"]
    D --> E["Store vectors<br/>with metadata"]
    F["User query"] --> G["Create query vector"]
    G --> H["Compare query<br/>to documents"]
    E --> H
    H --> I["Return ranked results"]
```


## Score Interpretation

The scores above are specific to this corpus and vectorizer. The same score can be strong in one setup and weak in another, depending on the documents, preprocessing choices, query style, and chunk length.

For real projects, **inspect ranked examples** instead of choosing a threshold blindly. Look at successful, ambiguous, and unrelated queries, and cases where the relevant document appears at rank 2 or rank 3.


## Compare Related And Unrelated Queries

Embedding systems always return a nearest neighbor, even for unrelated queries. That is convenient if queries are in-scope, but without a fallback rule the system cannot distinguish a strong match from a weak one.


In [ ]:
# An unrelated query is included to show why nearest-neighbor search needs fallback policy.
queries = [
    "How do I count tokens?",
    "How do I estimate LLM cost?",
    "How do I bake bread?",
]
for query in queries:
    scores = similarity_scores(corpus, query)
    best_index = max(range(len(scores)), key=lambda index: scores[index])
    print(f"{query!r} -> {records[best_index]['title']} ({scores[best_index]:.3f})")

A simple fallback checks whether the top score exceeds a threshold: below it, the system should return no result or ask for clarification instead. Notebook 04 covers this in practice.


## What To Inspect

Do not treat the score as a universal confidence value. Compare top-1, top-3, and failure cases. Check whether the result actually helps answer the query.


## Production Translation

In production, the vectorizer might be an embedding API and the vectors might live in a vector database. The same debugging questions still apply:
- Which text was embedded?
- Which metadata was stored?
- What query was embedded?
- What were the top-k results?
- Did the result actually answer the question?

Vector search should be **observable**, because surprising retrieval results are normal during development.


## When To Use A Library Instead Of This Local Vectorizer

Use the local vectorizer when the goal is inspection, learning, or a tiny reproducible example. Move to a library when you need better semantic matching, multilingual support, larger document collections, persistence, metadata filtering, or fast nearest-neighbor search. A practical progression is: first prove the retrieval workflow locally, then replace only the vectorization or storage layer with a production tool while keeping the same evaluation questions.


## Common Embedding Failure Modes

| Failure mode | What it looks like | First fix to try |
| --- | --- | --- |
| Keyword mismatch | Relevant result does not appear because wording differs. | Try a stronger embedding model or add clearer document text. |
| Semantic drift | Result is broadly related but not useful. | Improve chunking and evaluate top-k examples. |
| Chunk boundary issue | Answer is split across chunks. | Add overlap or use section-aware chunking. |
| Weak metadata | Result cannot be traced or cited. | Store source, title, section, and update date. |

The goal is to retrieve context that helps the next system step make a grounded decision.


## Practice

Edit `your_query` and rerun the cell. Try one attention question, one hallucination question, and one unrelated question.


In [ ]:
# @TODO: Change `your_query` and rerun the cell to inspect how rankings change.

your_query = "How are word embeddings trained?"
your_scores = similarity_scores(corpus, your_query)
your_ranked = sorted(
    range(len(your_scores)), key=lambda index: your_scores[index], reverse=True
)
for rank, index in enumerate(your_ranked[:3], start=1):
    print(f"{rank}. {records[index]['title']} | score={your_scores[index]:.3f}")

### Practice Reference

Course-related queries should retrieve relevant modules, while an unrelated query still receives a nearest neighbor. Keep in mind that similarity search ranks candidates; it does not decide whether the query belongs in scope or if the match is good. A production workflow therefore needs score inspection, source review, and fallback behavior.

<details>
  <summary>Show Examples</summary>

  1. Attention question:
   
     ```python
     your_query = "How does attention help a transformer handle long input?"
     ```

  2. Hallucination question:
   
     ```python
     your_query = "How do I detect hallucination in a generated answer?"
     ```

  3. Unrelated question:
   
     ```python
     your_query = "What is the highest point in Uruguay?"
     ```
     
  </details>


## Check Your Understanding

- What does an embedding represent?
- Why is cosine similarity useful for text retrieval?
- Why can an unrelated query still return a result?
- What would you inspect before trusting an embedding search system?

<details>
<summary>Show Reference Answers</summary>

- An embedding represents text as a numeric vector whose position captures patterns useful for comparison.
- Cosine similarity is useful because it compares vector direction, which helps rank texts that are semantically close even when their length differs.
- An unrelated query still returns a result because nearest-neighbor search ranks the closest available document even when every candidate is weak.
- Inspect the top-k results, shared terms or explanations when available, score distribution, source metadata, failure cases, and whether a fallback threshold is needed.

</details>
